# Taiwanese Bankruptcy 10/40/50 Setup

Build the canonical **10/40/50 (SVD / Train / Test)** split for the Taiwanese Bankruptcy Prediction dataset (UCI ID 572), train the reference MLP classifier on the 40% partition, and compute the clean SVD basis on the 10% partition.

Output directory: `experiments/data/taiwan_bankruptcy_10_40_50/`.

Follows the same conventions as the Spambase setup notebooks (`spambase_train_test_setup.ipynb` + `spambase_resplit_setup.ipynb`), collapsed into a single notebook because no legacy 50/50 split exists for this dataset.

## 1. Setup & Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from ucimlrepo import fetch_ucirepo
import warnings

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)

RANDOM_STATE = 42
OUTPUT_DIR = os.path.join('..', '..', 'data', 'taiwan_bankruptcy_10_40_50')
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')

/data/home/yluo147/myenv/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Using device: cuda
GPU: NVIDIA RTX A6000
CUDA Version: 12.4


## 2. Load & Preprocess Taiwanese Bankruptcy

The raw UCI dataset contains 95 features (94 continuous financial ratios + 1 binary indicator `Net Income Flag`). `Net Income Flag` is constant (always 1) across all 6819 samples, which would (a) collapse to 0 under MinMax scaling, (b) yield an undefined correlation with the target, and (c) contribute a zero singular value. Following the project convention of using only informative features, we drop it before splitting, leaving **94 features**. Column names also carry leading whitespace in the UCI archive — we strip it.

In [2]:
taiwan_bankruptcy = fetch_ucirepo(id=572)
X_raw = taiwan_bankruptcy.data.features.copy()
y_raw = taiwan_bankruptcy.data.targets.copy()

X_raw.columns = [c.strip() for c in X_raw.columns]
y_raw.columns = [c.strip() for c in y_raw.columns]

constant_cols = [c for c in X_raw.columns if X_raw[c].nunique() <= 1]
print(f'Constant feature columns to drop: {constant_cols}')
X_raw = X_raw.drop(columns=constant_cols)

df = pd.concat([X_raw, y_raw], axis=1)
target_col = y_raw.columns[0]
feature_names = X_raw.columns.tolist()

print(f'Dataset: {taiwan_bankruptcy.metadata["name"]}')
print(f'Samples: {df.shape[0]}, Features: {len(feature_names)} (raw 95, dropped {len(constant_cols)} constant)')
print(f'Target column: {target_col!r}')
print(f'Class distribution:\n{df[target_col].value_counts().to_string()}')
print(f'Positive rate: {df[target_col].mean():.4%}')

Constant feature columns to drop: ['Net Income Flag']
Dataset: Taiwanese Bankruptcy Prediction
Samples: 6819, Features: 94 (raw 95, dropped 1 constant)
Target column: 'Bankrupt?'
Class distribution:
Bankrupt?
0    6599
1     220
Positive rate: 3.2263%


In [3]:
scaler = MinMaxScaler()
df[feature_names] = scaler.fit_transform(df[feature_names])

weights = abs(df.corr()[target_col]).drop(target_col).values
weights = np.nan_to_num(weights, nan=0.0)
weights = weights / (np.linalg.norm(weights) + 1e-8)

bounds = [df[feature_names].min().values, df[feature_names].max().values]

print(f'Scaling: MinMaxScaler to [0, 1]')
print(f'Feature weights shape: {weights.shape}, L2 norm: {np.linalg.norm(weights):.6f}')
print(f'Top-5 weighted features:')
for idx in np.argsort(weights)[-5:][::-1]:
    print(f'  {feature_names[idx]}: weight={weights[idx]:.4f}')
print(f'Bounds shape: {[b.shape for b in bounds]}')

Scaling: MinMaxScaler to [0, 1]
Feature weights shape: (94,), L2 norm: 1.000000
Top-5 weighted features:
  Net Income to Total Assets: weight=0.2831
  ROA(A) before interest and % after tax: weight=0.2540
  ROA(B) before interest and depreciation after tax: weight=0.2451
  ROA(C) before interest and depreciation before interest: weight=0.2341
  Net worth/Assets: weight=0.2245
Bounds shape: [(94,), (94,)]


## 3. 10/40/50 Stratified Split

Following the canonical protocol established in Entry 13 of the experiment log: 10% held-out for SVD basis, 40% training, 50% test. Stratified by the (imbalanced) bankruptcy label. Done in two stages so the test split is fixed first, matching the Spambase resplit logic.

In [4]:
X_all = df[feature_names].values.astype(np.float64)
y_all = df[target_col].values.astype(int)

X_trainsvd, X_test, y_trainsvd, y_test = train_test_split(
    X_all, y_all,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_all,
)

X_train, X_svd, y_train, y_svd = train_test_split(
    X_trainsvd, y_trainsvd,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_trainsvd,
)

n_total = X_all.shape[0]
print(f'SVD partition  (10%): {X_svd.shape},  pos rate: {y_svd.mean():.4%}')
print(f'Train partition (40%): {X_train.shape}, pos rate: {y_train.mean():.4%}')
print(f'Test partition (50%): {X_test.shape},  pos rate: {y_test.mean():.4%}')
print(f'Fractions: {X_svd.shape[0] / n_total:.4f} / {X_train.shape[0] / n_total:.4f} / {X_test.shape[0] / n_total:.4f}')
assert X_svd.shape[0] + X_train.shape[0] + X_test.shape[0] == n_total

SVD partition  (10%): (682, 94),  pos rate: 3.2258%
Train partition (40%): (2727, 94), pos rate: 3.2270%
Test partition (50%): (3410, 94),  pos rate: 3.2258%
Fractions: 0.1000 / 0.3999 / 0.5001


## 4. Compute Clean SVD Basis on the 10% Partition

In [5]:
U_svd, S_svd, Vt_svd = np.linalg.svd(X_svd, full_matrices=False)

total_energy = (S_svd ** 2).sum()
print(f'U_svd:  {U_svd.shape}')
print(f'S_svd:  {S_svd.shape}')
print(f'Vt_svd: {Vt_svd.shape}')
print(f'Top-5 singular values: {S_svd[:5]}')
print(f'Energy in top-1:  {(S_svd[:1] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-5:  {(S_svd[:5] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-10: {(S_svd[:10] ** 2).sum() / total_energy:.2%}')
print(f'Energy in top-20: {(S_svd[:20] ** 2).sum() / total_energy:.2%}')

U_svd:  (682, 94)
S_svd:  (94,)
Vt_svd: (94, 94)
Top-5 singular values: [110.42671634  10.7300855    9.91424476   9.1363602    7.58909981]
Energy in top-1:  94.77%
Energy in top-5:  97.52%
Energy in top-10: 99.20%
Energy in top-20: 99.89%


## 5. Define & Train MLP Classifier on the 40% Training Partition

Same 3-layer architecture as `SpambaseNet`, adjusted to the Taiwan Bankruptcy feature count (`D_in = 94`).

In [6]:
class TaiwanBankruptcyNet(nn.Module):
    def __init__(self, D_in):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1),
        )

    def forward(self, x):
        return self.layer(x)


torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

D_in = X_train.shape[1]
model = TaiwanBankruptcyNet(D_in).to(device)
print(model)

TaiwanBankruptcyNet(
  (layer): Sequential(
    (0): Linear(in_features=94, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=2, bias=True)
    (5): Softmax(dim=-1)
  )
)


In [7]:
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.nn.functional.one_hot(
    torch.LongTensor(y_train), num_classes=2
).float().to(device)

X_test_t = torch.FloatTensor(X_test).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
N_EPOCHS = 100

print('Training...')
model.train()
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS} - Loss: {loss.item():.4f}')

model.eval()
print('Training complete.')

Training...
  Epoch  20/100 - Loss: 0.4024
  Epoch  40/100 - Loss: 0.1429
  Epoch  60/100 - Loss: 0.1425
  Epoch  80/100 - Loss: 0.1378
  Epoch 100/100 - Loss: 0.1371
Training complete.


## 6. Evaluate on Train and Test Sets

The dataset is severely imbalanced (~3.2% positive). A baseline that predicts the majority class achieves ~96.8% accuracy. Report both overall accuracy and per-class metrics so the imbalance is explicit; this is consistent with the Spambase pattern and serves the same role (a reference classifier for downstream attack experiments, not a tuned model).

In [8]:
with torch.no_grad():
    train_preds = model(X_train_t).argmax(dim=1).cpu().numpy()
    test_preds = model(X_test_t).argmax(dim=1).cpu().numpy()

train_acc = (train_preds == y_train).mean()
test_acc = (test_preds == y_test).mean()

print(f'Train Accuracy: {train_acc:.2%}')
print(f'Test Accuracy:  {test_acc:.2%}')
print()
print('Test Set Classification Report:')
print(classification_report(y_test, test_preds, target_names=['Non-Bankrupt', 'Bankrupt'], zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(y_test, test_preds))

Train Accuracy: 96.77%
Test Accuracy:  96.77%

Test Set Classification Report:
              precision    recall  f1-score   support

Non-Bankrupt       0.97      1.00      0.98      3300
    Bankrupt       0.00      0.00      0.00       110

    accuracy                           0.97      3410
   macro avg       0.48      0.50      0.49      3410
weighted avg       0.94      0.97      0.95      3410

Confusion Matrix:
[[3300    0]
 [ 110    0]]


## 7. Save Artifacts

In [9]:
np.savez(
    os.path.join(OUTPUT_DIR, 'train_test_data.npz'),
    X_svd=X_svd, y_svd=y_svd,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
)

np.savez(
    os.path.join(OUTPUT_DIR, 'preprocessing.npz'),
    weights=weights,
    bounds_min=bounds[0],
    bounds_max=bounds[1],
)

np.savez(
    os.path.join(OUTPUT_DIR, 'clean_svd_basis.npz'),
    U_svd=U_svd, S_svd=S_svd, Vt_svd=Vt_svd,
)

metadata = {
    'dataset': 'Taiwanese Bankruptcy Prediction (UCI ID 572)',
    'split_ratio': '10/40/50 (SVD/Train/Test)',
    'random_state': RANDOM_STATE,
    'stratified': True,
    'n_features': D_in,
    'feature_names': feature_names,
    'target_name': target_col,
    'dropped_constant_features': constant_cols,
    'n_svd': int(X_svd.shape[0]),
    'n_train': int(X_train.shape[0]),
    'n_test': int(X_test.shape[0]),
    'class_distribution_total': {
        '0': int((y_all == 0).sum()),
        '1': int((y_all == 1).sum()),
    },
    'class_distribution_svd': {
        '0': int((y_svd == 0).sum()),
        '1': int((y_svd == 1).sum()),
    },
    'class_distribution_train': {
        '0': int((y_train == 0).sum()),
        '1': int((y_train == 1).sum()),
    },
    'class_distribution_test': {
        '0': int((y_test == 0).sum()),
        '1': int((y_test == 1).sum()),
    },
    'scaler': 'MinMaxScaler [0, 1]',
}
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'TaiwanBankruptcyNet',
    'D_in': D_in,
    'hidden_layers': [64, 32],
    'n_classes': 2,
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'optimizer': 'Adam',
    'lr': 0.001,
    'epochs': N_EPOCHS,
    'loss': 'BCELoss',
    'trained_on': '40% partition of taiwan_bankruptcy_10_40_50',
}, os.path.join(OUTPUT_DIR, 'taiwan_bankruptcy_mlp.pth'))

print(f'Artifacts saved to {OUTPUT_DIR}/:')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(fpath)
    print(f'  {fname} ({size:,} bytes)')

Artifacts saved to ../../data/taiwan_bankruptcy_10_40_50/:
  clean_svd_basis.npz (585,054 bytes)
  metadata.json (3,943 bytes)
  preprocessing.npz (3,028 bytes)
  taiwan_bankruptcy_mlp.pth (36,180 bytes)
  train_test_data.npz (5,183,926 bytes)


## 8. Summary

All artifacts saved under `experiments/data/taiwan_bankruptcy_10_40_50/`:

| File | Contents |
|------|----------|
| `train_test_data.npz` | `X_svd`, `y_svd`, `X_train`, `y_train`, `X_test`, `y_test` (numpy arrays, scaled) |
| `preprocessing.npz` | `weights` (L2-normalized correlation weights), `bounds_min`, `bounds_max` |
| `clean_svd_basis.npz` | `U_svd`, `S_svd`, `Vt_svd` from the 10% SVD partition |
| `metadata.json` | Split configuration, feature names, per-partition class distributions |
| `taiwan_bankruptcy_mlp.pth` | Trained `TaiwanBankruptcyNet` state dict + hyperparameters |